In [1]:
from pathlib import Path
import json

import pymupdf


# --------------------------------------------------
# 1. PROJECT PATHS
# --------------------------------------------------

def find_project_root(start: Path) -> Path:
    """Walk up from the current folder until we find the repo root."""
    for candidate in (start, *start.parents):
        if (candidate / "data" / "input").is_dir():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

PDF_PATH = PROJECT_ROOT / "data" / "input" / "sample_input_1.pdf"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "extraction"

PAGE_IMAGES_DIR = OUTPUT_DIR / "page_images"
EXTRACTED_IMAGES_DIR = OUTPUT_DIR / "embedded_images"


# --------------------------------------------------
# 2. HELPER: MAKE DATA JSON-SAFE
# --------------------------------------------------

def json_safe(value):
    """Convert tuples and nested containers into JSON-safe values."""
    if isinstance(value, (tuple, list)):
        return [json_safe(item) for item in value]

    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}

    return value


# --------------------------------------------------
# 3. EXTRACT TEXT AND FONT INFORMATION
# --------------------------------------------------

def extract_text_blocks(page):
    """Extract text spans with their positions and font information."""
    page_dict = page.get_text("dict", sort=True)
    extracted_blocks = []

    for block in page_dict["blocks"]:
        if block.get("type") != 0:
            continue

        for line in block.get("lines", []):
            for span in line.get("spans", []):
                text = span.get("text", "")

                if not text.strip():
                    continue

                extracted_blocks.append({
                    "text": text,
                    "bbox": json_safe(span.get("bbox")),
                    "font": span.get("font"),
                    "font_size": span.get("size"),
                    "font_flags": span.get("flags"),
                    "color": span.get("color"),
                })

    return extracted_blocks


# --------------------------------------------------
# 4. SAVE EMBEDDED IMAGES
# --------------------------------------------------

def extract_embedded_images(document, page):
    """Save images referenced by a page, without duplicating xrefs."""
    image_xrefs = []

    for image_info in page.get_images(full=True):
        xref = image_info[0]

        if xref > 0 and xref not in image_xrefs:
            image_xrefs.append(xref)

    saved_images = []

    for xref in image_xrefs:
        image_info = document.extract_image(xref)

        if not image_info:
            continue

        extension = image_info["ext"]
        image_path = EXTRACTED_IMAGES_DIR / f"image_{xref}.{extension}"

        if not image_path.exists():
            image_path.write_bytes(image_info["image"])

        saved_images.append({
            "xref": xref,
            "path": str(image_path.relative_to(PROJECT_ROOT)),
            "width": image_info["width"],
            "height": image_info["height"],
            "format": extension,
        })

    return saved_images


# --------------------------------------------------
# 5. INSPECT THE COMPLETE PDF
# --------------------------------------------------

def inspect_pdf(pdf_path):
    if not pdf_path.exists():
        raise FileNotFoundError(
            f"PDF not found: {pdf_path}\n"
            "Place your sample PDF at data/input/sample_input_1.pdf."
        )

    PAGE_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    EXTRACTED_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

    all_pages = []
    all_text = []
    all_image_objects = {}

    with pymupdf.open(pdf_path) as document:
        if document.needs_pass:
            raise ValueError(
                "This PDF is password-protected. "
                "Provide an authorized, accessible copy."
            )

        if len(document) == 0:
            raise ValueError("The PDF has no pages.")

        print(f"PDF: {pdf_path.name}")
        print(f"Pages: {len(document)}")

        for page_index, page in enumerate(document):
            page_number = page_index + 1

            # A. Extract selectable/native text.
            text = page.get_text("text", sort=True).strip()
            text_blocks = extract_text_blocks(page)

            # B. Extract images referenced by this page.
            images = extract_embedded_images(document, page)

            for image in images:
                all_image_objects[image["xref"]] = image

            # C. Render the whole page for visual inspection.
            render_path = PAGE_IMAGES_DIR / f"page_{page_number:03}.png"

            pixmap = page.get_pixmap(
                matrix=pymupdf.Matrix(1.5, 1.5),
                alpha=False,
            )
            pixmap.save(str(render_path))

            # D. Record page-level metadata.
            if len(text) >= 40:
                text_status = "native_text_likely_available"
            else:
                text_status = "possible_scan_or_sparse_page"

            page_record = {
                "page_number": page_number,
                "width_points": page.rect.width,
                "height_points": page.rect.height,
                "rotation": page.rotation,
                "text_character_count": len(text),
                "text_status": text_status,
                "text": text,
                "text_blocks": text_blocks,
                "embedded_images": images,
                "rendered_page": str(render_path.relative_to(PROJECT_ROOT)),
            }

            all_pages.append(page_record)

            all_text.append(
                f"\n{'=' * 20} PAGE {page_number} {'=' * 20}\n\n{text}\n"
            )

            print(
                f"Page {page_number}/{len(document)} | "
                f"Characters: {len(text)} | "
                f"Image objects: {len(images)} | "
                f"Status: {text_status}"
            )

    # --------------------------------------------------
    # 6. SAVE RESULTS
    # --------------------------------------------------

    inventory = {
        "source_pdf": str(Path(pdf_path).resolve().relative_to(PROJECT_ROOT)),
        "page_count": len(all_pages),
        "unique_embedded_image_count": len(all_image_objects),
        "pages": all_pages,
    }

    inventory_path = OUTPUT_DIR / "page_inventory.json"
    text_path = OUTPUT_DIR / "all_text.txt"
    report_path = OUTPUT_DIR / "inspection_report.md"

    inventory_path.write_text(
        json.dumps(inventory, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    text_path.write_text(
        "\n".join(all_text),
        encoding="utf-8",
    )

    possible_scans = [
        page["page_number"]
        for page in all_pages
        if page["text_status"] == "possible_scan_or_sparse_page"
    ]

    report = [
        "# PDF Inspection Report",
        "",
        f"- Source PDF: `{pdf_path.name}`",
        f"- Pages: {len(all_pages)}",
        f"- Unique embedded image objects: {len(all_image_objects)}",
        f"- Pages needing text inspection: {possible_scans or 'None flagged'}",
        "",
        "## Important limitation",
        "",
        "Low extracted text can indicate a scan, a graphic-heavy page, "
        "or a sparse page. It does not prove that OCR is required.",
        "Review the rendered page images before deciding on OCR.",
        "",
        "## Page summary",
        "",
        "| Page | Characters | Image objects | Initial status |",
        "|---:|---:|---:|---|",
    ]

    for page in all_pages:
        report.append(
            f"| {page['page_number']} "
            f"| {page['text_character_count']} "
            f"| {len(page['embedded_images'])} "
            f"| {page['text_status']} |"
        )

    report_path.write_text(
        "\n".join(report),
        encoding="utf-8",
    )

    print("\nInspection complete.")
    print(f"Inventory: {inventory_path}")
    print(f"Extracted text: {text_path}")
    print(f"Report: {report_path}")
    print(f"Rendered pages: {PAGE_IMAGES_DIR}")
    print(f"Embedded images: {EXTRACTED_IMAGES_DIR}")


inspect_pdf(PDF_PATH)


PDF: sample_input_1.pdf
Pages: 5


Page 1/5 | Characters: 7283 | Image objects: 2 | Status: native_text_likely_available


Page 2/5 | Characters: 5201 | Image objects: 2 | Status: native_text_likely_available


Page 3/5 | Characters: 8240 | Image objects: 2 | Status: native_text_likely_available


Page 4/5 | Characters: 5899 | Image objects: 2 | Status: native_text_likely_available


Page 5/5 | Characters: 8549 | Image objects: 3 | Status: native_text_likely_available

Inspection complete.
Inventory: D:\Acalvio\DeCraft\outputs\extraction\page_inventory.json
Extracted text: D:\Acalvio\DeCraft\outputs\extraction\all_text.txt
Report: D:\Acalvio\DeCraft\outputs\extraction\inspection_report.md
Rendered pages: D:\Acalvio\DeCraft\outputs\extraction\page_images
Embedded images: D:\Acalvio\DeCraft\outputs\extraction\embedded_images
